In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST DRIVERS FILE
# ============================================================
# Purpose:
# 1. Read drivers.json using Spark DataFrame Reader API
# 2. Apply an explicit schema including nested JSON structure
# 3. Add ingestion metadata
# 4. Write the data into a Bronze Delta table
# ============================================================


# COMMAND ----------

# Load Environment Configuration

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# Load Bronze Helper Functions

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# ============================================================
# Define Source File and Target Table
# ============================================================

source_file = f"{landing_folder_path}/drivers.json"

table_name = (
    f"{catalog_name}.{schema_name}.{bronze_prefix}_drivers"
)

print("Source File :", source_file)
print("Target Table:", table_name)


# COMMAND ----------

# ============================================================
# Step 1 - Define Schema for drivers.json
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType
)

drivers_schema = StructType([

    StructField(
        "driverId",
        StringType(),
        True
    ),

    StructField(
        "name",
        StructType([
            StructField("givenName", StringType(), True),
            StructField("familyName", StringType(), True)
        ]),
        True
    ),

    StructField(
        "dateOfBirth",
        DateType(),
        True
    ),

    StructField(
        "nationality",
        StringType(),
        True
    ),

    StructField(
        "url",
        StringType(),
        True
    )

])


# COMMAND ----------

# ============================================================
# Step 2 - Read drivers.json
# ============================================================

drivers_df = (
    spark.read
    .format("json")
    .option("dateFormat", "yyyy-MM-dd")
    .schema(drivers_schema)
    .load(source_file)
)

display(drivers_df)


# COMMAND ----------

# Check DataFrame Schema

drivers_df.printSchema()


# COMMAND ----------

# ============================================================
# Step 3 - Add Ingestion Metadata
# ============================================================

drivers_final_df = add_ingestion_metadata(drivers_df)

display(drivers_final_df)


# COMMAND ----------

# ============================================================
# Step 4 - Write to Bronze Delta Table
# ============================================================

(
    drivers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


# COMMAND ----------

# ============================================================
# Step 5 - Validate Bronze Delta Table
# ============================================================

display(
    spark.table(table_name)
)